In [1]:
import random
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import torch.optim.lr_scheduler as lr_scheduler

# ==========================================
# 1. DATASET GENERATION (Parallel Logic)
# ==========================================
def calculate_cost(sequence):
    """
    Simulates the 3 servers in parallel.
    S1 adds 2 with 'A' and 1 with 'C'.
    S2 adds 3 with 'B'.
    S3 adds 1 with 'C'.
    The total cost is the maximum of the three (the bottleneck).
    """
    s1, s2, s3 = 0.0, 0.0, 0.0
    for char in sequence:
        if char == 'A':
            s1 += 2.0
        elif char == 'B':
            s2 += 3.0
        elif char == 'C':
            s1 += 1.0
            s3 += 1.0
            
    return float(max(s1, s2, s3))

# Generate 10,000 random words of varying lengths
data = []
for _ in range(10000):
    length = random.randint(1, 10)
    sequence = "".join(random.choices(['A', 'B', 'C'], k=length))
    data.append({'sequence': sequence, 'cost': calculate_cost(sequence)})

df = pd.DataFrame(data)

# ==========================================
# 2. DATA PREPARATION
# ==========================================
vocab = {'A': 1, 'B': 2, 'C': 3}

class SequenceDataset(Dataset):
    def __init__(self, dataframe):
        self.X = [torch.tensor([vocab[char] for char in seq], dtype=torch.long) for seq in dataframe['sequence']]
        self.Y = torch.tensor(dataframe['cost'].values, dtype=torch.float32)

    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.Y[idx]

def collate_fn(batch):
    X, Y = zip(*batch)
    lengths = torch.tensor([len(x) for x in X], dtype=torch.long)
    X_pad = pad_sequence(X, batch_first=True, padding_value=0)
    return X_pad, torch.stack(Y), lengths

dataset = SequenceDataset(df)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)

# ==========================================
# 3. MODEL 
# ==========================================
class CostPredictorRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super(CostPredictorRNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        # 32 hidden neurons should be enough to isolate the 3 counters
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True, nonlinearity='relu')
        self.linear = nn.Linear(hidden_dim, 1)

    def forward(self, x, lengths):
        emb = self.embedding(x)
        packed_emb = nn.utils.rnn.pack_padded_sequence(emb, lengths, batch_first=True, enforce_sorted=False)
        _, h_n = self.rnn(packed_emb)
        prediction = self.linear(h_n[-1, :, :])
        return prediction.squeeze()

model = CostPredictorRNN(vocab_size=4, embed_dim=8, hidden_dim=32)

# MAE (L1Loss) is ideal for this problem because we want absolute precision
criterion = nn.L1Loss() 
optimizer = optim.Adam(model.parameters(), lr=0.01)
scheduler = lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, verbose=True)

# ==========================================
# 4. TRAINING
# ==========================================
epochs = 50 
print("Starting RNN training (Parallel Cluster)...")

for epoch in range(epochs):
    model.train()
    total_loss = 0
    
    for X_batch, Y_batch, lengths in train_loader:
        optimizer.zero_grad()
        predictions = model(X_batch, lengths)
        loss = criterion(predictions, Y_batch)
        loss.backward()
        
        # Prevent ReLU from exploding (Crucial when summing parallel values)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        total_loss += loss.item()
    
    avg_loss = total_loss / len(train_loader)
    scheduler.step(avg_loss)
    
    if (epoch + 1) % 10 == 0 or epoch == 0:
        current_lr = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch+1:03d}/{epochs} - Loss (MAE): {avg_loss:.4f} | LR: {current_lr:.6f}")

# ==========================================
# 5. QUICK TEST
# ==========================================
print("\nTesting predictions (Network vs Reality):")
model.eval()
# Examples designed to see how the bottleneck changes
examples = [
    "AAAA",   # S1 wins (8.0)
    "BB",     # S2 wins (6.0)
    "AABB",   # S2 wins (4 vs 6 = 6.0)
    "BBAAA",  # S1 wins (6 vs 6 = 6.0, if there is another A it will be 8.0)
    "CCC",    # S1/S3 tie (3.0)
    "ABACAB"  # Combined test
]

with torch.no_grad():
    for seq in examples:
        tensor_in = torch.tensor([vocab[c] for c in seq], dtype=torch.long).unsqueeze(0)
        length = torch.tensor([len(seq)], dtype=torch.long)
        pred = model(tensor_in, length).item()
        actual_cost = calculate_cost(seq)
        print(f"Sequence: {seq:<7} | RNN predicts: {pred:.4f} | Actual cost: {actual_cost:.1f}")

C:\Users\rafam\anaconda3\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Starting RNN training (Parallel Cluster)...
Epoch 001/50 - Loss (MAE): 1.5845 | LR: 0.010000
Epoch 010/50 - Loss (MAE): 0.2867 | LR: 0.010000
Epoch 020/50 - Loss (MAE): 0.1934 | LR: 0.010000
Epoch 030/50 - Loss (MAE): 0.1672 | LR: 0.010000
Epoch 040/50 - Loss (MAE): 0.0650 | LR: 0.005000
Epoch 050/50 - Loss (MAE): 0.0724 | LR: 0.005000

Testing predictions (Network vs Reality):
Sequence: AAAA    | RNN predicts: 7.9285 | Actual cost: 8.0
Sequence: BB      | RNN predicts: 6.0379 | Actual cost: 6.0
Sequence: AABB    | RNN predicts: 5.9720 | Actual cost: 6.0
Sequence: BBAAA   | RNN predicts: 5.9646 | Actual cost: 6.0
Sequence: CCC     | RNN predicts: 2.9200 | Actual cost: 3.0
Sequence: ABACAB  | RNN predicts: 6.8763 | Actual cost: 7.0


In [2]:
import torch
import sys
import os

from QuantitativeObservationTable import (
    QuantitativeObservationTable, 
    QuantitativeObservationTableParameters
)

from TrainedRNNOracle import TrainedRNNOracle

oracle = TrainedRNNOracle(rnn_model=model, vocab_dict=vocab, tol=0.8, rounding_step=1)

params = QuantitativeObservationTableParameters(
    tol_dist_init=0.8, 
    rec_method="direct", 
    r=2, 
)

table = QuantitativeObservationTable(
    alphabet=oracle.alphabet,
    membership_query=oracle.calculate_weight, 
    parameters=params
)

seen_counterexamples = set()
iteration = 1

while True:
    print(f"\n--- Iteration {iteration} ---")
    table.make_table_closed_and_consistent()
            
    # Reconstruct Hypothesis WFA
    wfa = table.reconstruct_WFA()
    wfa.check_twins_property()
    
    # Equivalence Query
    ce = oracle.equivalence_query(
        wfa, 
        method="hybrid", 
        exhaustive_len=10, 
        random_max_len=15, 
        num_random=0,
    )

    # Evaluate Equivalence Query Results
    if ce is None:
        print(f"\n[SUCCESS] No counterexamples found.")
        print(f"Extraction complete in {iteration} iterations. Total states: {len(table.S)}")
        break

    if ce in seen_counterexamples:
        print(f"\n[STOP] Repeated counterexample '{ce}' detected. Check tolerances or network precision.")
        break

    print(f"[+] Processing Counterexample: '{ce}' | S size: {len(table.S)}")

    # Process Counterexample
    table.process_counterexample(ce, hypothesis_wfa=wfa, method="all_suffixes")
    seen_counterexamples.add(ce)
            
    iteration += 1


--- Iteration 1 ---
[OK] The Automaton satisfies the Twins Property. It is 100% determinizable!
  [EQ-Hybrid] Phase 1: Exhaustive search up to length 10...
      -> Failure detail for 'AB': RNN=3.00, WFA=5.00

[!] Counterexample found (Exhaustive): 'AB'
[+] Processing Counterexample: 'AB' | S size: 1
[Counterexample] Added ALL suffixes to E from: 'AB'

--- Iteration 2 ---
[!] Property Failed: Pair (0, 0) and (1, 0) accumulate infinite delay.
  [EQ-Hybrid] Phase 1: Exhaustive search up to length 10...
      -> Failure detail for 'BA': RNN=3.00, WFA=5.00

[!] Counterexample found (Exhaustive): 'BA'
[+] Processing Counterexample: 'BA' | S size: 3
[Counterexample] Added ALL suffixes to E from: 'BA'

--- Iteration 3 ---
[!] Property Failed: Pair (0, 0) and (3, 0) accumulate infinite delay.
  [EQ-Hybrid] Phase 1: Exhaustive search up to length 10...
      -> Failure detail for 'AAABB': RNN=6.00, WFA=8.00

[!] Counterexample found (Exhaustive): 'AAABB'
[+] Processing Counterexample: 'AAABB' 

In [3]:
wfa=table.apply_algebraic_minimization(wfa)

[Algebraic Minimization] Pruning state dimension space from 11 down to 2...


In [22]:
from graphviz import Digraph
import numpy as np

def draw_tropical_wfa(wfa: 'TropicalWFA', name="TropicalWFA"):
    """
    Generates a Graphviz diagram for a TropicalWFA object,
    formatting weights to a maximum of 3 decimal places.
    """
    dot = Digraph(name)
    dot.attr(rankdir='LR') # Left-to-right orientation
    
    n_states = len(wfa.q0)
    
    # 1. Create nodes (states) and initial/final transitions
    for i in range(n_states):
        label = f"S{i}"
        
        # Process final weights (beta)
        is_final = wfa.final[i] != float('-inf')
        if is_final:
            # Se aplica round() para limitar a 3 decimales
            label += f"\n[β={round(wfa.final[i], 3)}]"
            # Double circle for final states
            dot.node(str(i), label, shape="doublecircle")
        else:
            # Normal circle for the rest
            dot.node(str(i), label, shape="circle")
        
        # Process initial weights (alpha) - Incoming arrows
        if wfa.q0[i] != float('-inf'):
            # Invisible ghost node for the incoming arrow
            dot.node(f"start{i}", "", shape="none", width="0", height="0")
            # Se aplica round() al peso inicial
            dot.edge(f"start{i}", str(i), label=f"α={round(wfa.q0[i], 3)}")

    # 2. Create internal transitions (delta)
    for char in wfa.alphabet:
        matrix = wfa.delta[char]
        for i in range(n_states):
            for j in range(n_states):
                weight = matrix[i, j]
                # If the weight is not -inf, there is a transition
                if weight != float('-inf'):
                    # Se aplica round() al peso de la transición
                    dot.edge(str(i), str(j), label=f"{char}:{round(weight, 3)}")
                    
    return dot


In [ ]:
dot = draw_tropical_wfa(wfa)
dot.render('NoDetNoNoiseCase2_Vit', view=True,format='png')

In [7]:
#Weight pushing
wfa.push_weights_to_positive()
wfa.normalize_global_bias()
wfa.push_final_weights_to_zero()


[+] Weight Pushing applied: Negative weights pushed towards the endpoints.
[+] Bias Normalized: Global weight shifted by 1.00.
[+] Backward Weight Pushing: Final weights absorbed to 0.0.


In [ ]:
import itertools
words = ["".join(p) for l in range(1, 10) for p in itertools.product(wfa.alphabet, repeat=l)]

wfa.prune_by_viterbi(words)

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import torch
import random
import matplotlib.pyplot as plt
import numpy as np

def evaluate_error_by_length(model, vocab_dict, max_length=30, samples_per_length=100):
    """
    Evaluates the mean absolute error of the RNN against the Oracle, 
    rounding the prediction to the nearest integer, for different sequence lengths.
    """
    model.eval()
    lengths = list(range(1, max_length + 1, 2))
    mean_errors = []

    print(f"Evaluating model up to sequence length {max_length}...")
    
    with torch.no_grad():
        for L in lengths:
            accumulated_error = 0.0
            
            for _ in range(samples_per_length):
                # 1. Generate random sequence
                seq = "".join(random.choices(['A', 'B', 'C'], k=L))
                actual_cost = calculate_cost(seq)
                
                # 2. RNN Prediction
                tensor_in = torch.tensor([vocab_dict[c] for c in seq], dtype=torch.long).unsqueeze(0)
                length_tensor = torch.tensor([L], dtype=torch.long)
                
                pred_raw = model(tensor_in, length_tensor).item()
                
                # 3. Round to nearest integer
                rounded_pred = round(pred_raw)
                
                # 4. Calculate Absolute Error
                abs_error = abs(rounded_pred - actual_cost)
                accumulated_error += abs_error
            
            # Save the Mean Absolute Error (MAE)
            mean_error = accumulated_error / samples_per_length
            mean_errors.append(mean_error)
            
    # ==========================================
    # VISUALIZATION
    # ==========================================
    plt.figure(figsize=(10, 6))

    # Error Plot (Red Line)
    color = 'tab:red'
    plt.plot(lengths, mean_errors, marker='o', color=color, linewidth=2, label='Mean Absolute Error')
    
    # Shade the training zone (Lengths 1 to 10)
    plt.axvspan(1, 10, color='gray', alpha=0.2, label='Training Zone (L <= 10)')

    # Formatting axes and titles in English
    plt.xlabel('Sequence Length', fontsize=12, fontweight='bold')
    plt.ylabel('Mean Absolute Error', fontsize=12, fontweight='bold')
    plt.title('RNN Degradation on Long Sequences', fontsize=14, fontweight='bold')
    
    # Set Y-axis from 0 to 1 (Uncomment if you want a fixed scale)
    #plt.ylim(0, 1.0)
    
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.xticks(np.arange(1, max_length + 1, step=5)) # X-axis ticks
    plt.legend(loc='upper left')
    
    plt.tight_layout()
  
    # Save the plot in high resolution
    plt.savefig('rnn_degradation.png', dpi=300, bbox_inches='tight')
    print("Plot saved as 'rnn_degradation.png'")
    # -----------------------

    plt.show()

# --- EXECUTION ---
# Call the function with your trained model
evaluate_error_by_length(model=model, vocab_dict=vocab, max_length=100, samples_per_length=500)

In [6]:
import random
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import torch.optim.lr_scheduler as lr_scheduler

# ==========================================
# 1. DATASET GENERATION (PARALLEL NOISE)
# ==========================================
def calculate_parallel_costs(sequence):
    """
    Simulates 3 servers in parallel. 
    Each operation suffers its own independent stochastic delay.
    """
    s1_clean, s2_clean, s3_clean = 0.0, 0.0, 0.0
    s1_noisy, s2_noisy, s3_noisy = 0.0, 0.0, 0.0
    
    for char in sequence:
        if char == 'A':
            # Server 1 works
            mu = 2.0
            sigma = 0.2 + (0.15 * mu)
            s1_clean += mu
            s1_noisy += max(0.0, random.gauss(mu, sigma))
            
        elif char == 'B':
            # Server 2 works
            mu = 3.0
            sigma = 0.2 + (0.15 * mu)
            s2_clean += mu
            s2_noisy += max(0.0, random.gauss(mu, sigma))
            
        elif char == 'C':
            # Server 1 and Server 3 work (independent)
            mu_c = 1.0
            sigma_c = 0.2 + (0.15 * mu_c)
            
            s1_clean += mu_c
            s1_noisy += max(0.0, random.gauss(mu_c, sigma_c))
            
            s3_clean += mu_c
            s3_noisy += max(0.0, random.gauss(mu_c, sigma_c))
            
    # The real bottleneck and the noisy bottleneck
    clean_cost = max(s1_clean, s2_clean, s3_clean)
    noisy_cost = max(s1_noisy, s2_noisy, s3_noisy)
    
    return float(noisy_cost), float(clean_cost)

# Generate 25,000 random sequences
data = []
for _ in range(25000):
    length = random.randint(1, 10)
    sequence = "".join(random.choices(['A', 'B', 'C'], k=length))
    c_noisy, c_clean = calculate_parallel_costs(sequence)
    
    data.append({
        'sequence': sequence, 
        'noisy_cost': c_noisy,       # Target for the RNN
        'clean_real_cost': c_clean   # Ground truth
    })

df = pd.DataFrame(data)

print("Dataset Sample (Bottleneck Noise):")
try:
    display(df.head(5))
except NameError:
    print(df.head(5))

# ==========================================
# 2. DATA PREPARATION
# ==========================================
vocab = {'A': 1, 'B': 2, 'C': 3}

class ParallelSequencesDataset(Dataset):
    def __init__(self, dataframe):
        self.X = [torch.tensor([vocab[char] for char in seq], dtype=torch.long) for seq in dataframe['sequence']]
        self.Y = torch.tensor(dataframe['noisy_cost'].values, dtype=torch.float32)

    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.Y[idx]

def collate_fn(batch):
    X, Y = zip(*batch)
    lengths = torch.tensor([len(x) for x in X], dtype=torch.long)
    X_pad = pad_sequence(X, batch_first=True, padding_value=0)
    return X_pad, torch.stack(Y), lengths

dataset = ParallelSequencesDataset(df)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)

# ==========================================
# 3. MODEL 
# ==========================================
class CostPredictorRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super(CostPredictorRNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        # Slightly increased to 64 neurons to handle the parallel 'max' operation well
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True, nonlinearity='relu')
        self.linear = nn.Linear(hidden_dim, 1)

    def forward(self, x, lengths):
        emb = self.embedding(x)
        packed_emb = nn.utils.rnn.pack_padded_sequence(emb, lengths, batch_first=True, enforce_sorted=False)
        _, h_n = self.rnn(packed_emb)
        prediction = self.linear(h_n[-1, :, :])
        return prediction.squeeze()

model = CostPredictorRNN(vocab_size=4, embed_dim=8, hidden_dim=64)
criterion = nn.L1Loss() 
optimizer = optim.Adam(model.parameters(), lr=0.01)
scheduler = lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, verbose=True)

# ==========================================
# 4. TRAINING
# ==========================================
epochs = 200 
print("\nStarting training (Distilling the Parallel Maximum)...")

for epoch in range(epochs):
    model.train()
    total_train_loss = 0
    
    for X_batch, Y_batch, lengths in train_loader:
        optimizer.zero_grad()
        predictions = model(X_batch, lengths)
        if predictions.dim() == 0: predictions = predictions.unsqueeze(0)
        loss = criterion(predictions, Y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_train_loss += loss.item()
    
    avg_train_loss = total_train_loss / len(train_loader)
    
    model.eval()
    total_test_loss = 0
    with torch.no_grad():
        for X_batch_test, Y_batch_test, lengths_test in test_loader:
            test_predictions = model(X_batch_test, lengths_test)
            if test_predictions.dim() == 0: test_predictions = test_predictions.unsqueeze(0)
            test_loss = criterion(test_predictions, Y_batch_test)
            total_test_loss += test_loss.item()
            
    avg_test_loss = total_test_loss / len(test_loader)
    scheduler.step(avg_test_loss)
    
    if (epoch + 1) % 10 == 0 or epoch == 0:
        current_lr = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch+1:03d}/{epochs} - Train MAE: {avg_train_loss:.4f} | Test MAE: {avg_test_loss:.4f} | LR: {current_lr:.6f}")

# ==========================================
# 5. TEST (The Triumph over the Noisy Bottleneck)
# ==========================================
print("\nTesting predictions (Network vs Mathematical Reality):")
model.eval()
examples = [
    "AAAA",   # S1 wins (8.0)
    "BB",     # S2 wins (6.0)
    "AABB",   # S2 wins (4 vs 6 = 6.0)
    "BBAAA",  # S1 wins (6 vs 6 = 6.0, if there is another A it will be 8.0)
    "CCC",    # S1/S3 tie (3.0)
    "ABACAB"  # Combined test (Max(6.0, 6.0, 2.0) = 6.0)
]

with torch.no_grad():
    for seq in examples:
        tensor_in = torch.tensor([vocab[c] for c in seq], dtype=torch.long).unsqueeze(0)
        length = torch.tensor([len(seq)], dtype=torch.long)
        
        pred = model(tensor_in, length).item()
        _, clean_real = calculate_parallel_costs(seq)
        
        print(f"Sequence: {seq:<7} | Clean Truth: {clean_real:5.1f} | RNN Prediction: {pred:5.1f}")

Dataset Sample (Bottleneck Noise):


,sequence,noisy_cost,clean_real_cost
0,CCBBCBCAA,8.256242,9.0
1,BBCCBCBCB,14.649519,15.0
2,BAC,3.686828,3.0
3,CBACC,5.153761,5.0
4,BBABACC,11.809607,9.0


KeyboardInterrupt: 

In [34]:
import torch
import sys
import os

from QuantitativeObservationTable import (
    QuantitativeObservationTable, 
    QuantitativeObservationTableParameters
)

from TrainedRNNOracle import TrainedRNNOracle

oracle = TrainedRNNOracle(rnn_model=model, vocab_dict=vocab, tol=0.8, rounding_step=0)

params = QuantitativeObservationTableParameters(
    tol_dist_init=0.8, 
    rec_method="direct", 
    r=2, 
)

table = QuantitativeObservationTable(
    alphabet=oracle.alphabet,
    membership_query=oracle.calculate_weight, 
    parameters=params
)

seen_counterexamples = set()
iteration = 1

while True:
    print(f"\n--- Iteration {iteration} ---")
    table.make_table_closed_and_consistent()
            
    # Reconstruct Hypothesis WFA
    wfa = table.reconstruct_WFA()
    wfa.check_twins_property()
    
    # Equivalence Query
    ce = oracle.equivalence_query(
        wfa, 
        method="hybrid", 
        exhaustive_len=3, 
        random_max_len=15, 
        num_random=0,
    )

    # Evaluate Equivalence Query Results
    if ce is None:
        print(f"\n[SUCCESS] No counterexamples found.")
        print(f"Extraction complete in {iteration} iterations. Total states: {len(table.S)}")
        break

    if ce in seen_counterexamples:
        print(f"\n[STOP] Repeated counterexample '{ce}' detected. Check tolerances or network precision.")
        break

    print(f"[+] Processing Counterexample: '{ce}' | S size: {len(table.S)}")

    # Process Counterexample
    table.process_counterexample(ce, hypothesis_wfa=wfa, method="all_suffixes")
    seen_counterexamples.add(ce)
            
    iteration += 1


--- Iteration 1 ---
[OK] The Automaton satisfies the Twins Property. It is 100% determinizable!
  [EQ-Hybrid] Phase 1: Exhaustive search up to length 3...
      -> Failure detail for 'AB': RNN=3.01, WFA=4.99

[!] Counterexample found (Exhaustive): 'AB'
[+] Processing Counterexample: 'AB' | S size: 1
[Counterexample] Added ALL suffixes to E from: 'AB'

--- Iteration 2 ---
[!] Property Failed: Pair (0, 0) and (0, 1) accumulate infinite delay.
  [EQ-Hybrid] Phase 1: Exhaustive search up to length 3...
      -> Failure detail for 'BA': RNN=2.98, WFA=4.94

[!] Counterexample found (Exhaustive): 'BA'
[+] Processing Counterexample: 'BA' | S size: 3
[Counterexample] Added ALL suffixes to E from: 'BA'

--- Iteration 3 ---
[!] Property Failed: Pair (0, 0) and (0, 1) accumulate infinite delay.
  [EQ-Hybrid] Phase 1: Exhaustive search up to length 3...
  [EQ-Hybrid] No counterexamples found. Hypothesis accepted!

[SUCCESS] No counterexamples found.
Extraction complete in 3 iterations. Total state

In [38]:
#Weight pushing
wfa.push_weights_to_positive()
wfa.normalize_global_bias()
wfa.push_final_weights_to_zero()


[+] Weight Pushing applied: Negative weights pushed towards the endpoints.
[+] Bias Normalized: Global weight shifted by 1.14.
[+] Backward Weight Pushing: Final weights absorbed to 0.0.


In [36]:
wfa=table.apply_algebraic_minimization(wfa)

[Algebraic Minimization] Pruning state dimension space from 5 down to 2...


In [44]:
import itertools
words = ["".join(p) for l in range(1, 4) for p in itertools.product(wfa.alphabet, repeat=l)]

wfa.prune_by_viterbi(words)

Starting Viterbi pruning with 39 sequences...
[+] Transition pruning: Removed 6 useless edges.
[+] State cleanup: All 2 states were useful. None removed.


In [46]:
dot = draw_tropical_wfa(wfa)
dot.render('NoDetCase2_direct', view=True,format='png')

'NoDetCase2_direct.png'

In [ ]:

ruta_guardado = "modelo_costes_rnn2.pth"

torch.save(model.state_dict(), ruta_guardado)

In [10]:
modelo_recuperado = CostPredictorRNN(vocab_size=4, embed_dim=8, hidden_dim=64)

modelo_recuperado.load_state_dict(torch.load("modelo_costes_rnn2.pth"))


modelo_recuperado.eval()

model = modelo_recuperado

C:\Users\rafam\AppData\Local\Temp\ipykernel_9184\1266731392.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  modelo_recuperado.load_state_dict(torch.load("modelo_costes_r